# Multimodal probe design — EEG / fNIRS / fDCS

Interactive walkthrough of the probe design pipeline developed for the Bachelor's Thesis
*Development of a multimodal strategy to investigate speech processing mechanisms in infants*
(Universitat Pompeu Fabra, 2024/2025).

The notebook loads each layout from `configs/`, reports its admissible source-detector
pairs and renders it to scale.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from probe_design import ProbeLayout, plot_layout

CONFIG_DIR = ROOT / 'configs'
sorted(p.name for p in CONFIG_DIR.glob('*.json'))


## 1. Design constraints

Every layout has to satisfy three families of constraints simultaneously:

| Constraint | Value | Why |
|---|---|---|
| fNIRS source–detector separation | 1.8–3.0 cm | separation sets the depth the measurement is weighted towards |
| fDCS source–detector separation | 1.5–2.0 cm | longer separations return far fewer photons |
| Clearance between a pair and other components | 0.25–0.30 cm | components packed too tightly leave no room between a pair |
| Component footprint | 0.7–1.6 cm across | optodes, electrodes and the pressure-mechanism housing all compete for space |

The footprints are the effective area each component occupies on the probe surface.
For the fDCS optodes that area is set by the housing of the pressure mechanism, not by
the optode itself.

Doing this by hand means redrawing the probe and recomputing every separation for each
candidate. The module below does it in milliseconds.


In [ ]:
layout = ProbeLayout.from_json(CONFIG_DIR / 'lateral_final.json')
print(layout.report())


## 2. Lateral probe — design iterations

Three lateral probes were printed. The first was built around a spring-loaded pressure
mechanism. Moving to the thread-actuated mechanism meant enlarging the fDCS openings
from 0.9 cm to 1.5 cm, which drove the second layout. The second forced the optic fibres
to bend, so the third was widened by 8 mm to allow straight routing without giving up
temporal-lobe coverage.


In [ ]:
for stem in ['lateral_v1', 'lateral_v2', 'lateral_final']:
    probe = ProbeLayout.from_json(CONFIG_DIR / f'{stem}.json')
    plot_layout(probe)


## 3. Occipital probe (control region)

The occipital lobe is not expected to activate during language tasks in typically
developing infants, so it serves as the control region and carries a low optode density
by design. Its separations are matched to the lateral probe, which reduces acquisition
geometry as a potential confound when the two regions are compared.

The physical probe also carries a rectangular accelerometer for motion monitoring. Its
coordinates are not recorded in the source notebook, so it is not represented here.


In [ ]:
probe = ProbeLayout.from_json(CONFIG_DIR / 'occipital.json')
print(probe.report())
plot_layout(probe)


## 4. Comparing separations across probes

The check that matters before fabrication: do the two final probes sample at the same
source-detector separations?


In [ ]:
for stem in ['lateral_final', 'occipital']:
    probe = ProbeLayout.from_json(CONFIG_DIR / f'{stem}.json')
    for channel in probe.all_channels():
        print(f'{probe.name:24s} {channel}')


## 5. Designing a new layout

Layouts are plain data, so exploring a new one is a matter of moving coordinates and
re-reading the report.


In [ ]:
candidate = ProbeLayout(
    name='Candidate probe',
    width=8.0,
    height=5.0,
    electrodes=[[0.7, 3.7], [0.7, 1.3], [7.3, 3.7], [7.3, 1.3]],
    fnirs_sources=[[2.4, 1.0], [5.6, 4.0]],
    fnirs_detectors=[[2.4, 3.9], [5.6, 1.1]],
    fdcs_sources=[[4.0, 2.5]],
    fdcs_detectors=[[4.0, 4.3], [5.8, 2.5], [4.0, 0.7], [2.2, 2.5]],
)
print(candidate.report())
plot_layout(candidate)
